# 04 — Genre Clustering & ENOA Space

Deep dive into the Every Noise at Once coordinate system, genre taxonomy,
spatial clustering, zone exploration, and cross-playlist genre overlap.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import polars as pl
import plotly.express as px
import plotly.graph_objects as go
import seaborn as sns
from sklearn.cluster import DBSCAN, KMeans

REPO_ROOT = Path("../..").resolve()
sys.path.insert(0, str(REPO_ROOT / "src"))

from etl.db import get_connection, init_schema
from recommend.modules.genre import (
    expand_genre_zone,
    filter_by_enoa_proximity,
    genre_to_enoa,
    load_genre_map_from_db,
)
from recommend.preprocessing import build_feature_matrix
from utils.logging import configure_logging

configure_logging()
sns.set_theme(style="whitegrid", palette="muted")

conn = get_connection(read_only=False)
init_schema(conn)
corpus = build_feature_matrix(conn)
genre_map = load_genre_map_from_db(conn)

## 1. ENOA Full Genre Map

Scatter of all 6k+ genres from `genre_xy` — `(left, top)`, colored by `color` column.

In [ ]:
# Load full genre_xy table
genre_xy = conn.execute('SELECT first_genre, top, "left", color FROM genre_xy').pl()
print(f"ENOA genres: {len(genre_xy):,}")

fig = px.scatter(
    genre_xy.to_pandas(),
    x="left", y="top",
    hover_name="first_genre",
    color="color",
    title=f"Every Noise at Once — Full Genre Map ({len(genre_xy):,} genres)",
    width=1000, height=800,
    opacity=0.6,
)
fig.update_yaxes(autorange="reversed")  # ENOA convention: top=0 at top
fig.update_layout(showlegend=False)
fig.show()

## 2. Library Overlay

Overlay library tracks' `(left, top)` onto the genre map.

In [ ]:
# Filter to tracks with real ENOA coords
library_enoa = corpus.filter(
    (pl.col("top") != 0) | (pl.col("left") != 0)
).select(["id", "track_name", "top", "left", "gen_4", "first_genre"])

print(f"Library tracks with ENOA coords: {len(library_enoa):,}")

fig = go.Figure()

# Background: all ENOA genres (light gray)
fig.add_trace(go.Scatter(
    x=genre_xy["left"].to_list(),
    y=genre_xy["top"].to_list(),
    mode="markers",
    marker=dict(size=2, color="lightgray", opacity=0.3),
    text=genre_xy["first_genre"].to_list(),
    hoverinfo="text",
    name="All ENOA genres",
))

# Foreground: library tracks colored by gen_4
gen4_groups = library_enoa.filter(pl.col("gen_4").is_not_null()).partition_by("gen_4", as_dict=True)
colors = {"acoustic": "#2ca02c", "dance": "#ff7f0e", "electronic": "#1f77b4", "instrumental": "#d62728"}

for (gen4_val,), group_df in gen4_groups.items():
    fig.add_trace(go.Scatter(
        x=group_df["left"].to_list(),
        y=group_df["top"].to_list(),
        mode="markers",
        marker=dict(size=4, color=colors.get(gen4_val, "purple"), opacity=0.6),
        text=group_df["track_name"].to_list(),
        hoverinfo="text",
        name=gen4_val,
    ))

fig.update_yaxes(autorange="reversed")
fig.update_layout(
    title="Library Tracks Overlaid on ENOA Genre Map",
    width=1000, height=800,
    xaxis_title="left", yaxis_title="top",
)
fig.show()

## 3. Genre Taxonomy Tree

Visualize the `gen_4 → gen_6 → gen_8 → my_genre` hierarchy.

In [ ]:
# Genre hierarchy from genre_map table
genre_hierarchy = conn.execute("""
    SELECT gen_4, gen_6, gen_8, my_genre, sub_genre, COUNT(*) as n_genres
    FROM genre_map
    WHERE gen_4 IS NOT NULL
    GROUP BY gen_4, gen_6, gen_8, my_genre, sub_genre
    ORDER BY gen_4, gen_6, gen_8, my_genre
""").pl()

print(f"Genre taxonomy entries: {len(genre_hierarchy)}")
genre_hierarchy.head(20)

In [ ]:
# Sunburst: gen_4 → gen_6 → gen_8
# Count library tracks per taxonomy level
tax_cols = ["gen_4", "gen_6", "gen_8"]
available_tax = [c for c in tax_cols if c in corpus.columns]

if len(available_tax) >= 2:
    tax_counts = (
        corpus.filter(pl.col(available_tax[0]).is_not_null())
        .group_by(available_tax)
        .len()
    ).to_pandas()

    fig = px.sunburst(
        tax_counts,
        path=available_tax,
        values="len",
        title="Genre Taxonomy — Library Track Distribution",
        width=800, height=800,
    )
    fig.show()
else:
    print("Insufficient taxonomy columns for sunburst")

## 4. Spatial Clustering on ENOA Coordinates

KMeans and DBSCAN on `(top, left)` for the 6k genre points.
Compare to the curated `gen_4` labels.

In [ ]:
# KMeans on genre_xy coordinates
genre_coords = genre_xy.select(["top", "left"]).to_numpy().astype(np.float64)

# Try k=4 (matching gen_4 cardinality)
for k in [4, 6, 8]:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(genre_coords)

    fig, ax = plt.subplots(figsize=(10, 8))
    scatter = ax.scatter(
        genre_coords[:, 1], genre_coords[:, 0],  # left, top
        c=labels, s=3, alpha=0.5, cmap="tab10",
    )
    ax.set_xlabel("left")
    ax.set_ylabel("top")
    ax.invert_yaxis()
    ax.set_title(f"KMeans k={k} on ENOA Genre Coordinates")
    plt.colorbar(scatter, ax=ax, label="Cluster")
    plt.tight_layout()
    plt.show()

In [ ]:
# DBSCAN — density-based clustering
# Normalize coordinates first since top and left have different scales
from sklearn.preprocessing import StandardScaler

coords_scaled = StandardScaler().fit_transform(genre_coords)
db = DBSCAN(eps=0.3, min_samples=10)
db_labels = db.fit_predict(coords_scaled)

n_clusters = len(set(db_labels)) - (1 if -1 in db_labels else 0)
n_noise = (db_labels == -1).sum()
print(f"DBSCAN: {n_clusters} clusters, {n_noise} noise points")

fig, ax = plt.subplots(figsize=(10, 8))
scatter = ax.scatter(
    genre_coords[:, 1], genre_coords[:, 0],
    c=db_labels, s=3, alpha=0.5, cmap="tab20",
)
ax.set_xlabel("left")
ax.set_ylabel("top")
ax.invert_yaxis()
ax.set_title(f"DBSCAN on ENOA (eps=0.3) — {n_clusters} clusters")
plt.tight_layout()
plt.show()

## 5. Genre Zone Exploration

Visualize `expand_genre_zone()` circles for selected genres.

In [ ]:
ZONE_GENRES = ["indie rock", "deep house", "bossa nova", "ambient"]
RADIUS = 1500.0

fig, axes = plt.subplots(2, 2, figsize=(16, 14))
axes = axes.flatten()

for i, genre_name in enumerate(ZONE_GENRES):
    ax = axes[i]
    center = genre_to_enoa(genre_name, genre_map)

    if center is None:
        ax.set_title(f"{genre_name} — not found in ENOA")
        continue

    # Background: all library tracks
    ax.scatter(
        library_enoa["left"].to_numpy(),
        library_enoa["top"].to_numpy(),
        s=1, alpha=0.1, color="gray",
    )

    # Zone circle
    circle = plt.Circle(
        (center[1], center[0]),  # (left, top)
        RADIUS, fill=False, color="red", linewidth=2, linestyle="--",
    )
    ax.add_patch(circle)

    # Tracks in zone
    zone_tracks = filter_by_enoa_proximity(corpus, center, radius=RADIUS)
    if not zone_tracks.is_empty():
        ax.scatter(
            zone_tracks["left"].to_numpy(),
            zone_tracks["top"].to_numpy(),
            s=8, alpha=0.6, color="red",
            label=f"{len(zone_tracks)} tracks in zone",
        )

    # Center point
    ax.scatter([center[1]], [center[0]], s=100, marker="*", color="gold", zorder=5)

    ax.set_xlabel("left")
    ax.set_ylabel("top")
    ax.invert_yaxis()
    ax.set_title(f"{genre_name} zone (r={RADIUS})")
    ax.legend(fontsize=8)
    ax.set_aspect("equal")

plt.suptitle("Genre Zone Exploration", fontsize=14)
plt.tight_layout()
plt.show()

## 6. Genre Coverage Gaps

Which gen_4/gen_6 regions are under-represented?

In [ ]:
# Track count per gen_4
if "gen_4" in corpus.columns:
    gen4_counts = corpus.group_by("gen_4").len().sort("len", descending=True)
    total = len(corpus)
    gen4_counts = gen4_counts.with_columns(
        (pl.col("len") / total * 100).round(1).alias("pct")
    )
    print("gen_4 coverage:")
    print(gen4_counts)

# gen_6 coverage
if "gen_6" in corpus.columns:
    gen6_counts = corpus.group_by("gen_6").len().sort("len", descending=True)
    gen6_counts = gen6_counts.with_columns(
        (pl.col("len") / total * 100).round(1).alias("pct")
    )
    print(f"\ngen_6 coverage ({len(gen6_counts)} groups):")
    gen6_counts

In [ ]:
# ENOA spatial coverage: divide coordinate space into grid, count tracks per cell
if "top" in corpus.columns and "left" in corpus.columns:
    grid_size = 500  # pixels per cell
    tracks_with_coords = corpus.filter((pl.col("top") != 0) | (pl.col("left") != 0))

    top_vals = tracks_with_coords["top"].to_numpy()
    left_vals = tracks_with_coords["left"].to_numpy()

    fig, ax = plt.subplots(figsize=(10, 8))
    h = ax.hist2d(left_vals, top_vals, bins=20, cmap="YlOrRd")
    plt.colorbar(h[3], ax=ax, label="Track count")
    ax.set_xlabel("left")
    ax.set_ylabel("top")
    ax.invert_yaxis()
    ax.set_title("ENOA Spatial Density — Library Tracks")
    plt.tight_layout()
    plt.show()

## 7. ENOA as Predictor of gen_4

How well do `(top, left)` coordinates predict the `gen_4` label?

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score

if "gen_4" in corpus.columns:
    # Filter to tracks with both ENOA coords and gen_4 label
    labeled = corpus.filter(
        pl.col("gen_4").is_not_null() &
        ((pl.col("top") != 0) | (pl.col("left") != 0))
    )
    print(f"Labeled tracks with ENOA coords: {len(labeled):,}")

    X = labeled.select(["top", "left"]).to_numpy()
    y = labeled["gen_4"].to_list()

    # Quick KNN cross-validation
    knn = KNeighborsClassifier(n_neighbors=15)
    scores = cross_val_score(knn, X, y, cv=5, scoring="accuracy")
    print(f"KNN (k=15) accuracy from (top, left) → gen_4: {scores.mean():.3f} ± {scores.std():.3f}")

    # Decision boundary visualization
    knn.fit(X, y)
    x_min, x_max = X[:, 1].min() - 100, X[:, 1].max() + 100
    y_min, y_max = X[:, 0].min() - 100, X[:, 0].max() + 100
    xx, yy = np.meshgrid(
        np.linspace(x_min, x_max, 200),
        np.linspace(y_min, y_max, 200),
    )
    # Note: model expects (top, left) but meshgrid gives (left, top)
    grid_points = np.c_[yy.ravel(), xx.ravel()]
    Z = knn.predict(grid_points)

    # Encode labels to ints for contour
    unique_labels = sorted(set(y))
    label_to_int = {l: i for i, l in enumerate(unique_labels)}
    Z_int = np.array([label_to_int[z] for z in Z]).reshape(xx.shape)
    y_int = np.array([label_to_int[yi] for yi in y])

    fig, ax = plt.subplots(figsize=(12, 10))
    ax.contourf(xx, yy, Z_int, alpha=0.3, cmap="tab10")
    scatter = ax.scatter(X[:, 1], X[:, 0], c=y_int, s=3, alpha=0.5, cmap="tab10")

    handles = [plt.Line2D([0], [0], marker='o', color='w',
               markerfacecolor=plt.cm.tab10(i / len(unique_labels)),
               markersize=8, label=l) for i, l in enumerate(unique_labels)]
    ax.legend(handles=handles, title="gen_4")
    ax.set_xlabel("left")
    ax.set_ylabel("top")
    ax.invert_yaxis()
    ax.set_title(f"KNN Decision Boundary: (top, left) → gen_4 (acc={scores.mean():.3f})")
    plt.tight_layout()
    plt.show()

## 8. Cross-Playlist Genre Overlap

Jaccard similarity of genre sets between playlists.

In [ ]:
# Build genre sets per playlist
playlist_genres = conn.execute("""
    SELECT p.playlist_name, tg.first_genre
    FROM playlist_tracks pt
    JOIN playlists p ON pt.playlist_id = p.playlist_id
    LEFT JOIN track_genre tg ON pt.track_id = tg.track_id
    WHERE tg.first_genre IS NOT NULL
""").pl()

if not playlist_genres.is_empty():
    genre_sets: dict[str, set[str]] = {}
    for name in playlist_genres["playlist_name"].unique().to_list():
        genres = set(
            playlist_genres.filter(pl.col("playlist_name") == name)["first_genre"].to_list()
        )
        genre_sets[name] = genres

    # Jaccard similarity matrix
    names = sorted(genre_sets.keys())
    n = len(names)
    jaccard = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            a, b = genre_sets[names[i]], genre_sets[names[j]]
            if len(a | b) > 0:
                jaccard[i, j] = len(a & b) / len(a | b)

    fig, ax = plt.subplots(figsize=(12, 10))
    sns.heatmap(
        jaccard,
        xticklabels=names, yticklabels=names,
        annot=True, fmt=".2f", cmap="YlOrRd",
        square=True, ax=ax,
    )
    ax.set_title("Cross-Playlist Genre Overlap (Jaccard Similarity on first_genre sets)")
    plt.tight_layout()
    plt.show()
else:
    print("No playlist genre data available")

In [ ]:
conn.close()